# StockMind AI: Model 1 — Supervised Stock Return Forecaster
### Reproducible Google Colab Free Training & Evaluation Pipeline

This notebook trains a self-contained supervised forecasting model to predict:
1. **Next-Trading-Day Return ($y_{1d}$):** $(Close_{t+1} / Close_t - 1) \times 100$
2. **Five-Trading-Day Return ($y_{5d}$):** $(Close_{t+5} / Close_t - 1) \times 100$

**Zero-Leakage & Governance Guarantees:**
- **Strict Chronological Splitting:** Train (2019-2023), Validation (2024), Test (2025-2026).
- **Isolated Preprocessing:** Imputers and scalers fit ONLY on training data.
- **Strict Symbol Grouping:** Rolling indicators and target shifts never bleed across stocks.
- **Trading Day Semantics:** Shift(5) represents 5 available trading observations, not calendar days.
- **Scientific Honesty:** Includes naive baselines and transaction-cost experiments without unsubstantiated profitability claims.


## 1. Environment & Path Configuration
Configure your data path below. In Google Colab Free, you can mount Google Drive or upload the `nse_daily/` folder directly.


In [ ]:
# Configuration Cell - Adjust paths as needed for Colab or Local
import os
from pathlib import Path

# Set to True if running in Google Colab with Google Drive mounted
MOUNT_GOOGLE_DRIVE = False
DRIVE_MOUNT_PATH = '/content/drive'

# Data Directory options:
# Option A (Google Drive): '/content/drive/MyDrive/stockmind/Dataset/market_data/nse_daily'
# Option B (Uploaded in Colab): '/content/nse_daily'
# Option C (Local fallback): relative to workspace
COLAB_DATA_DIR = '/content/nse_daily'
LOCAL_FALLBACK_DIR = 'Dataset/market_data/nse_daily'

if os.path.exists(COLAB_DATA_DIR):
    DATA_DIR = COLAB_DATA_DIR
elif os.path.exists(LOCAL_FALLBACK_DIR):
    DATA_DIR = LOCAL_FALLBACK_DIR
else:
    DATA_DIR = 'Dataset/market_data/nse_daily'

OUTPUT_DIR = 'artifacts/trained_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Universe Filter: Set to None for full 3,000+ universe, or specify top liquid tickers
# Using top liquid stocks keeps training under 60 seconds on free Colab CPU!
TARGET_STOCKS = [
    'RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK',
    'HINDUNILVR', 'ITC', 'SBIN', 'BHARTIARTL', 'KOTAKBANK',
    'LT', 'AXISBANK', 'ASIANPAINT', 'MARUTI', 'TITAN',
    'BAJFINANCE', 'SUNPHARMA', 'TATASTEEL', 'NTPC', 'POWERGRID'
]

# Chronological Split Dates
TRAIN_START = '2019-01-01'
TRAIN_END = '2023-12-31'
VAL_START = '2024-01-01'
VAL_END = '2024-12-31'
TEST_START = '2025-01-01'
TEST_END = '2026-10-23'

print(f'Using Data Directory: {DATA_DIR}')
print(f'Output Directory: {OUTPUT_DIR}')
print(f'Target Universe: {len(TARGET_STOCKS)} stocks' if TARGET_STOCKS else 'Full Universe')


## 2. Dependencies & Drive Mounting (Optional)


In [ ]:
# Mount Google Drive if configured
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT_PATH)
    print('Google Drive mounted.')

# Install required packages (Colab has most preinstalled)
!pip install -q pyarrow lightgbm scikit-learn joblib pandas numpy tabulate
print('Dependencies verified.')


## 3. Data Loading & Semantic Integrity Check
Loads official NSE Parquet files, verifies column data types, filters strictly to regular equities (`series == 'EQ'`), and reports excluded rows.


In [ ]:
import glob
import pandas as pd
import numpy as np

# Find parquet files
parquet_files = sorted(glob.glob(f'{DATA_DIR}/*.parquet'))
print(f'Found {len(parquet_files)} parquet files: {[os.path.basename(f) for f in parquet_files]}')

if not parquet_files:
    raise FileNotFoundError(f'No parquet files found in {DATA_DIR}. Please upload nse_daily files or mount Drive.')

columns_needed = ['date', 'symbol', 'series', 'open', 'high', 'low', 'close', 'prev_close', 'volume']

dfs = []
raw_total = 0
excluded_series = 0
excluded_symbols = 0

for pf in parquet_files:
    df_chunk = pd.read_parquet(pf, columns=columns_needed)
    raw_total += len(df_chunk)
    
    # Filter 1: Series == 'EQ' (Standard Continuous Equity Trading)
    mask_eq = df_chunk['series'] == 'EQ'
    excluded_series += int((~mask_eq).sum())
    df_chunk = df_chunk[mask_eq]
    
    # Filter 2: Universe filter (if enabled)
    if TARGET_STOCKS:
        sym_set = set(s.upper() for s in TARGET_STOCKS)
        mask_sym = df_chunk['symbol'].str.upper().isin(sym_set)
        excluded_symbols += int((~mask_sym).sum())
        df_chunk = df_chunk[mask_sym]
        
    dfs.append(df_chunk)

raw_df = pd.concat(dfs, ignore_index=True)
raw_df['date'] = pd.to_datetime(raw_df['date'])

# Semantic Deduplication & Chronological Sorting
raw_df = raw_df.drop_duplicates(subset=['symbol', 'date']).sort_values(['symbol', 'date']).reset_index(drop=True)

print('\n=== Data Audit & Semantic Report ===')
print(f'Total raw rows scanned: {raw_total:,}')
print(f'Excluded non-EQ rows (BE, BZ series): {excluded_series:,}')
print(f'Excluded out-of-universe rows: {excluded_symbols:,}')
print(f'Retained clean observations: {len(raw_df):,}')
print(f'Unique Symbols: {raw_df["symbol"].nunique()}')
print(f'Date Range: {raw_df["date"].min().date()} to {raw_df["date"].max().date()}')


## 4. Feature Engineering & Target Construction
Calculates backward technical indicators and constructs forward returns using the mandatory formulas:
- $\text{target\_1d} = (\text{grouped\_close.shift}(-1) / \text{close} - 1) \times 100$
- $\text{target\_5d} = (\text{grouped\_close.shift}(-5) / \text{close} - 1) \times 100$


In [ ]:
def compute_technical_features(df):
    """Calculates backward-looking indicators strictly within each symbol group."""
    df = df.sort_values(by=["symbol", "date"]).copy()
    chunks = []
    
    for sym, g in df.groupby("symbol", sort=False):
        g = g.copy()
        close = g["close"]
        open_ = g["open"]
        high = g["high"]
        low = g["low"]
        vol = g["volume"]
        
        # Backward Returns (%)
        g["ret_1d"] = (close / close.shift(1) - 1.0) * 100.0
        g["ret_2d"] = (close / close.shift(2) - 1.0) * 100.0
        g["ret_5d"] = (close / close.shift(5) - 1.0) * 100.0
        g["ret_10d"] = (close / close.shift(10) - 1.0) * 100.0
        g["ret_20d"] = (close / close.shift(20) - 1.0) * 100.0
        g["open_to_close_ret"] = (close / open_ - 1.0) * 100.0
        
        # Exponential Moving Averages
        ema_9 = close.ewm(span=9, adjust=False).mean()
        ema_21 = close.ewm(span=21, adjust=False).mean()
        ema_50 = close.ewm(span=50, adjust=False).mean()
        ema_200 = close.ewm(span=200, adjust=False).mean()
        
        g["ema_ratio_9"] = (close / ema_9 - 1.0) * 100.0
        g["ema_ratio_21"] = (close / ema_21 - 1.0) * 100.0
        g["ema_ratio_50"] = (close / ema_50 - 1.0) * 100.0
        g["ema_ratio_200"] = (close / ema_200 - 1.0) * 100.0
        g["trend_ema_50_200"] = (ema_50 / ema_200 - 1.0) * 100.0
        
        # RSI 14
        delta = close.diff()
        gain = delta.clip(lower=0)
        loss = (-delta).clip(lower=0)
        avg_gain = gain.rolling(window=14, min_periods=14).mean()
        avg_loss = loss.rolling(window=14, min_periods=14).mean()
        rs = avg_gain / (avg_loss.replace(0, np.nan))
        g["rsi_14"] = 100.0 - (100.0 / (1.0 + rs))
        
        # MACD (12, 26, 9)
        fast_ema = close.ewm(span=12, adjust=False).mean()
        slow_ema = close.ewm(span=26, adjust=False).mean()
        macd_line = fast_ema - slow_ema
        macd_sig = macd_line.ewm(span=9, adjust=False).mean()
        macd_hist = macd_line - macd_sig
        g["macd_line_pct"] = (macd_line / close) * 100.0
        g["macd_signal_pct"] = (macd_sig / close) * 100.0
        g["macd_hist_pct"] = (macd_hist / close) * 100.0
        
        # Volatility & NATR 14
        log_ret = np.log(close / close.shift(1))
        g["volatility_20d"] = log_ret.rolling(window=20, min_periods=10).std() * np.sqrt(252) * 100.0
        
        prev_close = close.shift(1)
        tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
        atr_14 = tr.rolling(window=14, min_periods=14).mean()
        g["natr_14"] = (atr_14 / close) * 100.0
        
        # Volume
        vol_5 = vol.rolling(window=5, min_periods=3).mean()
        vol_20 = vol.rolling(window=20, min_periods=10).mean()
        g["volume_ratio_5_20"] = vol_5 / (vol_20.replace(0, np.nan))
        g["log_volume"] = np.log1p(vol)
        
        # Range Position
        high_20 = high.rolling(window=20, min_periods=10).max()
        low_20 = low.rolling(window=20, min_periods=10).min()
        g["range_pos_20d"] = ((close - low_20) / (high_20 - low_20).replace(0, np.nan)).clip(0.0, 1.0)
        
        high_52w = high.rolling(window=252, min_periods=40).max()
        low_52w = low.rolling(window=252, min_periods=40).min()
        g["range_pos_52w"] = ((close - low_52w) / (high_52w - low_52w).replace(0, np.nan)).clip(0.0, 1.0)
        
        # Construct Targets (Mandatory formulas)
        g["target_1d"] = (close.shift(-1) / close - 1.0) * 100.0
        g["target_5d"] = (close.shift(-5) / close - 1.0) * 100.0
        
        chunks.append(g)
        
    return pd.concat(chunks, ignore_index=True)

feature_df = compute_technical_features(raw_df)
FEATURE_COLS = [
    'ret_1d', 'ret_2d', 'ret_5d', 'ret_10d', 'ret_20d', 'open_to_close_ret',
    'ema_ratio_9', 'ema_ratio_21', 'ema_ratio_50', 'ema_ratio_200', 'trend_ema_50_200',
    'rsi_14', 'macd_line_pct', 'macd_signal_pct', 'macd_hist_pct',
    'volatility_20d', 'natr_14', 'volume_ratio_5_20', 'log_volume',
    'range_pos_20d', 'range_pos_52w'
]

print(f'Calculated {len(FEATURE_COLS)} features across {len(feature_df):,} rows.')


## 5. Chronological Train / Validation / Test Splitting & Preprocessor Fitting
Scalers and imputers are fit **exclusively** on the Training partition to strictly prevent lookahead bias.


In [ ]:
from sklearn.preprocessing import RobustScaler

# Drop terminal rows where target is NaN (per symbol)
valid_df = feature_df.dropna(subset=['target_1d', 'target_5d']).copy()

train_mask = (valid_df['date'] >= pd.to_datetime(TRAIN_START)) & (valid_df['date'] <= pd.to_datetime(TRAIN_END))
val_mask = (valid_df['date'] >= pd.to_datetime(VAL_START)) & (valid_df['date'] <= pd.to_datetime(VAL_END))
test_mask = (valid_df['date'] >= pd.to_datetime(TEST_START)) & (valid_df['date'] <= pd.to_datetime(TEST_END))

train_df = valid_df[train_mask].reset_index(drop=True)
val_df = valid_df[val_mask].reset_index(drop=True)
test_df = valid_df[test_mask].reset_index(drop=True)

print(f'Train split: {len(train_df):,} rows ({train_df["date"].min().date()} to {train_df["date"].max().date()})')
print(f'Val split:   {len(val_df):,} rows ({val_df["date"].min().date()} to {val_df["date"].max().date()})')
print(f'Test split:  {len(test_df):,} rows ({test_df["date"].min().date()} to {test_df["date"].max().date()})')

# Learn Imputation Medians STRICTLY from Training Set
train_medians = train_df[FEATURE_COLS].median().to_dict()

def prepare_X(df, medians):
    X = df[FEATURE_COLS].copy()
    for col, med in medians.items():
        X[col] = X[col].fillna(med)
    return X

X_train_raw = prepare_X(train_df, train_medians)
X_val_raw = prepare_X(val_df, train_medians)
X_test_raw = prepare_X(test_df, train_medians)

# Fit Scaler STRICTLY on Training Set
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

y1_train = train_df['target_1d'].values
y1_val = val_df['target_1d'].values
y1_test = test_df['target_1d'].values

y5_train = train_df['target_5d'].values
y5_val = val_df['target_5d'].values
y5_test = test_df['target_5d'].values

print('Preprocessing isolated and fitted successfully on training partition.')


## 6. Baselines Comparison (Zero-Return, Momentum, Ridge)
Establishing standard benchmarks before testing gradient-boosted trees.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge

def directional_accuracy(y_true, y_pred):
    """Calculates percentage of correct sign predictions."""
    correct_sign = (np.sign(y_true) == np.sign(y_pred)).sum()
    return float((correct_sign / len(y_true)) * 100.0)

def evaluate_predictions(y_true, y_pred, model_name="Model", horizon="1D"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    da = directional_accuracy(y_true, y_pred)
    return {"model": model_name, "horizon": horizon, "mae": round(mae, 4), "rmse": round(rmse, 4), "da_pct": round(da, 2)}

eval_results = []

# Baseline 1: Zero Return Persistence (predict 0.0%)
eval_results.append(evaluate_predictions(y1_val, np.zeros_like(y1_val), "Naive Zero Return", "1D"))
eval_results.append(evaluate_predictions(y5_val, np.zeros_like(y5_val), "Naive Zero Return", "5D"))

# Baseline 2: Momentum Persistence (predict previous 1D return)
ret_1d_val = val_df['ret_1d'].fillna(0).values
eval_results.append(evaluate_predictions(y1_val, ret_1d_val, "Momentum Persistence", "1D"))

# Baseline 3: Linear Ridge Regression
ridge_1d = Ridge(alpha=100.0).fit(X_train, y1_train)
ridge_5d = Ridge(alpha=100.0).fit(X_train, y5_train)
eval_results.append(evaluate_predictions(y1_val, ridge_1d.predict(X_val), "Ridge Regression", "1D"))
eval_results.append(evaluate_predictions(y5_val, ridge_5d.predict(X_val), "Ridge Regression", "5D"))

results_df = pd.DataFrame(eval_results)
print('=== Validation Set Baseline Benchmarks ===')
print(results_df.to_markdown(index=False))


## 7. Champion Model Training: LightGBM Regressor
LightGBM builds histogram-binned decision trees with early stopping monitored strictly on the validation partition.


In [ ]:
import lightgbm as lgb

print('Training LightGBM 1-Day Forecaster...')
lgb_1d = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=5,
    num_leaves=24,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
lgb_1d.fit(
    X_train, y1_train,
    eval_set=[(X_val, y1_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=25, verbose=False)]
)

print('Training LightGBM 5-Day Forecaster...')
lgb_5d = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=5,
    num_leaves=24,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
lgb_5d.fit(
    X_train, y5_train,
    eval_set=[(X_val, y5_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=25, verbose=False)]
)

eval_results.append(evaluate_predictions(y1_val, lgb_1d.predict(X_val), "LightGBM Champion", "1D"))
eval_results.append(evaluate_predictions(y5_val, lgb_5d.predict(X_val), "LightGBM Champion", "5D"))

updated_df = pd.DataFrame(eval_results)
print('\n=== Validation Comparison: Baselines vs Champion ===')
print(updated_df.to_markdown(index=False))


## 8. Final Out-of-Sample Evaluation on Held-Out Test Set (2025–2026)
Evaluating exclusively on the unseen test partition.


In [ ]:
test_eval = []

# Baselines on Test Set
test_eval.append(evaluate_predictions(y1_test, np.zeros_like(y1_test), "Naive Zero Return", "1D"))
test_eval.append(evaluate_predictions(y5_test, np.zeros_like(y5_test), "Naive Zero Return", "5D"))
test_eval.append(evaluate_predictions(y1_test, ridge_1d.predict(X_test), "Ridge Regression", "1D"))
test_eval.append(evaluate_predictions(y5_test, ridge_5d.predict(X_test), "Ridge Regression", "5D"))

# Champion LightGBM on Test Set
pred_1d_test = lgb_1d.predict(X_test)
pred_5d_test = lgb_5d.predict(X_test)
test_eval.append(evaluate_predictions(y1_test, pred_1d_test, "LightGBM Champion", "1D"))
test_eval.append(evaluate_predictions(y5_test, pred_5d_test, "LightGBM Champion", "5D"))

test_results_df = pd.DataFrame(test_eval)
print('=== FINAL OUT-OF-SAMPLE TEST EVALUATION (2025-2026) ===')
print(test_results_df.to_markdown(index=False))

# Per-Symbol Slice Evaluation (Top Equities)
test_df['pred_1d'] = pred_1d_test
test_df['pred_5d'] = pred_5d_test

symbol_evals = []
for sym, g in test_df.groupby('symbol'):
    da1 = directional_accuracy(g['target_1d'].values, g['pred_1d'].values)
    mae1 = mean_absolute_error(g['target_1d'].values, g['pred_1d'].values)
    symbol_evals.append({'symbol': sym, 'sample_count': len(g), 'da_1d_pct': round(da1, 2), 'mae_1d': round(mae1, 4)})

symbol_df = pd.DataFrame(symbol_evals).sort_values('sample_count', ascending=False)
print('\n=== Per-Symbol Out-of-Sample Performance (Sample) ===')
print(symbol_df.head(10).to_markdown(index=False))


## 9. Research Simulation: Transaction-Cost-Aware Filter
*Note: This is an academic research simulation, NOT proof of live execution profitability or financial advice.*


In [ ]:
# Research Simulation: Long-only filter when predicted 1D return > threshold + roundtrip slippage
ROUNDTRIP_COST_PCT = 0.10  # 10 bps slippage and exchange fees
ALPHA_THRESHOLD = 0.20     # Only enter if predicted return > 20 bps

test_df['long_signal'] = test_df['pred_1d'] > (ALPHA_THRESHOLD + ROUNDTRIP_COST_PCT)
trades = test_df[test_df['long_signal']]

if len(trades) > 0:
    gross_ret = trades['target_1d'].mean()
    net_ret = gross_ret - ROUNDTRIP_COST_PCT
    win_rate = (trades['target_1d'] > ROUNDTRIP_COST_PCT).mean() * 100.0
    print(f'Trades Filtered: {len(trades):,} out of {len(test_df):,} opportunities')
    print(f'Gross Average Realized Return: {gross_ret:+.3f}%')
    print(f'Net Average Realized Return (after 10 bps cost): {net_ret:+.3f}%')
    print(f'Win Rate after Transaction Cost: {win_rate:.1f}%')
else:
    print('No trades met the combined alpha and slippage threshold.')


## 10. Model Artifact & Schema Export
Exports model binaries, fitted preprocessor, feature schema, and evaluation summary.
PaperTradex can directly load these artifacts using `StockForecaster.load()`.


In [ ]:
import joblib
import json

# 1. Save Models
joblib.dump(lgb_1d, os.path.join(OUTPUT_DIR, 'model_1d.joblib'))
joblib.dump(lgb_5d, os.path.join(OUTPUT_DIR, 'model_5d.joblib'))

# 2. Save Preprocessor Object
preprocessor_payload = {
    'medians': train_medians,
    'scaler': scaler,
    'feature_names': FEATURE_COLS
}
joblib.dump(preprocessor_payload, os.path.join(OUTPUT_DIR, 'preprocessor.joblib'))

# 3. Save Feature Schema
schema_data = {
    'feature_names': FEATURE_COLS,
    'feature_count': len(FEATURE_COLS),
    'target_horizons': ['target_1d', 'target_5d'],
    'metadata': {
        'model_architecture': 'LightGBM-HistRegressor',
        'train_range': f'{TRAIN_START} to {TRAIN_END}',
        'val_range': f'{VAL_START} to {VAL_END}',
        'test_range': f'{TEST_START} to {TEST_END}',
    }
}
with open(os.path.join(OUTPUT_DIR, 'feature_schema.json'), 'w') as f:
    json.dump(schema_data, f, indent=2)

# 4. Save Final Evaluation Report
report_payload = {
    'validation_metrics': updated_df.to_dict(orient='records'),
    'test_metrics': test_results_df.to_dict(orient='records'),
    'top_symbols': symbol_df.head(10).to_dict(orient='records')
}
with open(os.path.join(OUTPUT_DIR, 'evaluation_report.json'), 'w') as f:
    json.dump(report_payload, f, indent=2)

print(f'All model artifacts successfully exported to: {OUTPUT_DIR}')
print('Files created:')
for f in os.listdir(OUTPUT_DIR):
    print(f'  - {f}')
